<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

# Document Loading

## ۱. ایده‌ی اصلی

Loader منبع (فایل/پوشه/URL) رو به یک شیء استاندارد `Document(doc_id, text, metadata)` تبدیل می‌کنه. بقیه‌ی پایپ‌لاین فقط با `Document` کار می‌کنه و از فرمت اصلی بی‌خبره.

## ۲. Document در برابر Chunk

| | Document | Chunk |
|---|---|---|
| سطح | کل فایل یا یک صفحه | تکه‌ی کوچک متن |
| ساخته می‌شه در | Loading | Chunking |

## ۳. Metadata ضروری

| فیلد | کاربرد |
|---|---|
| source | مسیر نسبی (citation) |
| file_type | نوع فایل |
| size_bytes | فیلتر فایل غیرعادی |
| modified_at | تازگی |
| content_hash | تشخیص تغییر و تکراری |
| encoding | عیب‌یابی متن خراب |

> `doc_id` از **مسیر** ساخته می‌شه (پایدار)، `content_hash` از **محتوا** (برای تشخیص تغییر).

## ۴. استراتژی Encoding

`utf-8-sig` ← `cp1256` ← `utf-8 (errors=replace)`

- `cp1256` تقریباً هیچ‌وقت خطا نمی‌ده، پس باید **بعد** از utf-8 امتحان بشه.
- همیشه encoding استفاده‌شده رو در metadata ذخیره کن.

## ۵. مدیریت خطا و LoadReport

| وضعیت | رفتار |
|---|---|
| پسوند ناشناخته | skip + ثبت |
| فایل خالی / فقط فاصله | skip + ثبت |
| بزرگ‌تر از حد مجاز | skip + ثبت |
| exception | ثبت در failed، ادامه‌ی کار |
| فایل مخفی | نادیده گرفتن |
| symlink بیرون از root | skip (امنیت) |

## ۶. الگوهای کلیدی

- **Loader Registry:** نگاشت پسوند ← Loader؛ افزودن فرمت جدید بدون تغییر بقیه‌ی کد
- **Lazy Loading:** generator به‌جای لیست کامل، برای حجم زیاد
- **Fail Threshold:** اگه درصد خطا زیاد بود، job شکست بخوره (alert)
- **Report-first:** هر اجرا یک `LoadReport` قابل مانیتور تولید کنه

## ۷. ایده‌ی کد این نوت‌بوک

- `Document`، `TextLoader` با تشخیص encoding و hash
- `DirectoryLoader` با registry، lazy load، skip مخفی/خالی/بزرگ/ناشناخته
- ساخت پوشه‌ی نمونه‌ی موقت با فایل‌های utf-8، BOM، cp1256 و فایل‌های مشکل‌دار
- نمایش `LoadReport`
- اثبات اینکه با تغییر محتوا، `doc_id` ثابت و `content_hash` عوض می‌شه

## ۸. چک‌لیست production

- [ ] Loader ها پشت یک اینترفیس مشترک و Registry هستن
- [ ] metadata در لحظه‌ی خواندن گرفته می‌شه
- [ ] `doc_id` پایدار (از مسیر) و `content_hash` (از محتوا) جدا هستن
- [ ] encoding مقاوم و ذخیره‌ی encoding استفاده‌شده
- [ ] یک فایل بد، کل job رو نمی‌خوابونه
- [ ] محدودیت اندازه، رد فایل مخفی و symlink بیرونی
- [ ] `LoadReport` لاگ می‌شه و آستانه‌ی خطا alert می‌ده
- [ ] برای حجم زیاد از lazy loading استفاده می‌شه

In [ ]:
import hashlib
import logging
import tempfile
from collections.abc import Iterator
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Protocol

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s | %(message)s", force=True
)
logger = logging.getLogger("loader")


# ======================================================================
# 1) مدل Document و اینترفیس Loader
# ======================================================================
@dataclass
class Document:
    doc_id: str
    text: str
    metadata: dict = field(default_factory=dict)


class Loader(Protocol):
    def load(self, path: Path) -> list[Document]: ...


# ======================================================================
# 2) رمزگشایی مقاوم (Encoding)
# ======================================================================
def decode_bytes(raw: bytes) -> tuple[str, str]:
    """utf-8-sig (شامل BOM) ← cp1256 ← utf-8 با جایگزینی. برمی‌گردونه: (متن، encoding)"""
    for enc in ("utf-8-sig", "cp1256"):
        try:
            return raw.decode(enc), enc
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace"), "utf-8(replace)"


# ======================================================================
# 3) TextLoader (txt و md)
# ======================================================================
class TextLoader:
    def __init__(self, file_type: str):
        self.file_type = file_type

    def load(self, path: Path) -> list[Document]:
        raw = path.read_bytes()
        text, encoding = decode_bytes(raw)
        stat = path.stat()
        metadata = {
            "source": str(path),  # DirectoryLoader به مسیر نسبی تبدیلش می‌کنه
            "file_name": path.name,
            "file_type": self.file_type,
            "size_bytes": stat.st_size,
            "modified_at": datetime.fromtimestamp(
                stat.st_mtime, tz=timezone.utc
            ).isoformat(),
            "content_hash": hashlib.sha256(raw).hexdigest(),
            "encoding": encoding,
        }
        return [Document(doc_id="", text=text, metadata=metadata)]


LOADERS: dict[str, Loader] = {
    ".txt": TextLoader("txt"),
    ".md": TextLoader("md"),
    # ".pdf": PDFLoader(...),   ← تاپیک بعدی
    # ".html": HTMLLoader(...), ← تاپیک بعدی
}


# ======================================================================
# 4) LoadReport و DirectoryLoader
# ======================================================================
@dataclass
class LoadReport:
    loaded: int = 0
    skipped: list = field(default_factory=list)  # [(مسیر، دلیل)]
    failed: list = field(default_factory=list)  # [(مسیر، خطا)]


class DirectoryLoader:
    def __init__(self, root, loaders: dict[str, Loader], max_bytes: int = 5_000_000):
        self.root = Path(root).resolve()
        self.loaders = loaders
        self.max_bytes = max_bytes
        self.report = LoadReport()

    def _candidate_files(self) -> Iterator[Path]:
        for p in sorted(self.root.rglob("*")):
            rel_parts = p.relative_to(self.root).parts
            if any(
                part.startswith(".") for part in rel_parts
            ):  # مخفی‌ها: .git, .env, ...
                continue
            if p.is_file():
                yield p

    def lazy_load(self) -> Iterator[Document]:
        self.report = LoadReport()
        for path in self._candidate_files():
            rel = path.relative_to(self.root).as_posix()

            # امنیت: symlink که به بیرون از root اشاره می‌کنه
            try:
                path.resolve().relative_to(self.root)
            except ValueError:
                self.report.skipped.append((rel, "outside_root"))
                continue

            loader = self.loaders.get(path.suffix.lower())
            if loader is None:
                self.report.skipped.append((rel, "unsupported_extension"))
                continue

            size = path.stat().st_size
            if size == 0:
                self.report.skipped.append((rel, "empty_file"))
                continue
            if size > self.max_bytes:
                self.report.skipped.append((rel, "too_large"))
                continue

            try:
                docs = loader.load(path)
            except Exception as e:  # یک فایل بد نباید کل کار رو بخوابونه
                self.report.failed.append((rel, repr(e)))
                logger.warning("failed to load %s: %r", rel, e)
                continue

            docs = [d for d in docs if d.text.strip()]
            if not docs:
                self.report.skipped.append((rel, "blank_text"))
                continue

            rel_id = hashlib.sha1(rel.encode()).hexdigest()[
                :12
            ]  # شناسه‌ی پایدار از مسیر
            for i, doc in enumerate(docs):
                doc.metadata["source"] = rel
                doc.doc_id = f"{rel_id}-{i}"
                self.report.loaded += 1
                yield doc

    def load(self) -> list[Document]:
        return list(self.lazy_load())


# ======================================================================
# 5) ساخت پوشه‌ی نمونه‌ی موقت (شامل فایل‌های مشکل‌دار)
# ======================================================================
def make_sample_tree(root: Path) -> None:
    (root / "hr").mkdir()
    (root / "legacy").mkdir()
    (root / ".hidden").mkdir()

    (root / "hr" / "leave_policy.md").write_text(
        "# مرخصی\nمرخصی استحقاقی ۳۰ روز در سال است.", encoding="utf-8"
    )
    (root / "hr" / "insurance.txt").write_bytes(
        ("\ufeff" + "بیمه تکمیلی هزینه‌های درمان را پوشش می‌دهد.").encode("utf-8")
    )
    (root / "legacy" / "old_notice.txt").write_bytes(
        "سلام به اداره".encode("cp1256")
    )  # encoding قدیمی
    (root / "hr" / "empty.txt").write_bytes(b"")  # خالی
    (root / "hr" / "blank.md").write_text("   \n\n  ", encoding="utf-8")  # فقط فاصله
    (root / "hr" / "big.txt").write_text("a" * 3000, encoding="utf-8")  # بزرگ‌تر از حد
    (root / "hr" / "sheet.xlsx").write_bytes(b"PK\x03\x04 fake")  # پسوند پشتیبانی‌نشده
    (root / ".hidden" / "secret.txt").write_text(
        "API_KEY=123", encoding="utf-8"
    )  # مخفی


# ======================================================================
# 6) اجرا
# ======================================================================
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    make_sample_tree(root)

    print("=" * 60, "\nالف) بارگذاری lazy (یکی‌یکی)\n" + "=" * 60)
    loader = DirectoryLoader(root, LOADERS, max_bytes=1000)
    docs = []
    for doc in loader.lazy_load():
        docs.append(doc)
        m = doc.metadata
        print(
            f"  {doc.doc_id} | {m['source']:<22} | enc={m['encoding']:<10} | hash={m['content_hash'][:8]} | {doc.text[:30]!r}"
        )

    print("\n" + "=" * 60, "\nب) LoadReport\n" + "=" * 60)
    r = loader.report
    print(f"  loaded : {r.loaded}")
    print(f"  skipped: {len(r.skipped)}")
    for path, reason in r.skipped:
        print(f"      - {path:<22} → {reason}")
    print(f"  failed : {len(r.failed)}")
    print("  (فایل مخفی .hidden/secret.txt اصلاً دیده نشد ✅)")

    print("\n" + "=" * 60, "\nج) doc_id ثابت، content_hash متغیر\n" + "=" * 60)
    before = {
        d.metadata["source"]: (d.doc_id, d.metadata["content_hash"]) for d in docs
    }

    (root / "hr" / "leave_policy.md").write_text(
        "# مرخصی\nمرخصی استحقاقی ۳۵ روز در سال است.", encoding="utf-8"
    )
    after = {
        d.metadata["source"]: (d.doc_id, d.metadata["content_hash"])
        for d in DirectoryLoader(root, LOADERS, max_bytes=1000).lazy_load()
    }

    key = "hr/leave_policy.md"
    print(
        f"  doc_id تغییر نکرد؟      {before[key][0] == after[key][0]}   ← می‌تونیم نسخه‌ی قدیمی رو پیدا و جایگزین کنیم"
    )
    print(
        f"  content_hash عوض شد؟    {before[key][1] != after[key][1]}   ← می‌فهمیم باید دوباره پردازش بشه"
    )
    unchanged = [
        k for k in before if k != key and k in after and before[k][1] == after[k][1]
    ]
    print(f"  فایل‌های بدون تغییر: {unchanged}  ← این‌ها دوباره پردازش نمی‌شن")

    print("\n" + "=" * 60, "\nد) آستانه‌ی شکست job\n" + "=" * 60)
    failed_ratio = len(r.failed) / max(1, r.loaded + len(r.failed))
    print(
        f"  نسبت خطا = {failed_ratio:.0%}  →",
        "🚨 job شکست می‌خوره" if failed_ratio > 0.05 else "✅ ادامه",
    )

الف) بارگذاری lazy (یکی‌یکی)
  3c997f352b16-0 | hr/insurance.txt       | enc=utf-8-sig  | hash=173caf80 | 'بیمه تکمیلی هزینه\u200cهای درمان را'
  bdd7ca36b29f-0 | hr/leave_policy.md     | enc=utf-8-sig  | hash=46ade735 | '# مرخصی\r\nمرخصی استحقاقی ۳۰ روز'
  2746ea964eba-0 | legacy/old_notice.txt  | enc=cp1256     | hash=b5ed3bde | 'سلام به اداره'

ب) LoadReport
  loaded : 3
  skipped: 4
      - hr/big.txt             → too_large
      - hr/blank.md            → blank_text
      - hr/empty.txt           → empty_file
      - hr/sheet.xlsx          → unsupported_extension
  failed : 0
  (فایل مخفی .hidden/secret.txt اصلاً دیده نشد ✅)

ج) doc_id ثابت، content_hash متغیر
  doc_id تغییر نکرد؟      True   ← می‌تونیم نسخه‌ی قدیمی رو پیدا و جایگزین کنیم
  content_hash عوض شد؟    True   ← می‌فهمیم باید دوباره پردازش بشه
  فایل‌های بدون تغییر: ['hr/insurance.txt', 'legacy/old_notice.txt']  ← این‌ها دوباره پردازش نمی‌شن

د) آستانه‌ی شکست job
  نسبت خطا = 0%  → ✅ ادامه
